# Rvector — Ring-Element Vector

A compact vector of elements in Z_{2^{ELL}}, ELL ∈ [1, 8].

**Construction**: ``Rvector(ell)(size)`` — two-step: factory then allocation.

**Operations**:
- ``v[i]`` / ``v[i] = val`` — read / write
- ``v.fill(val)``, ``v.rand_fill()`` — bulk fill
- ``v.batch_set(indices, val)`` — set multiple positions
- ``v.to_bytes()`` / ``v.from_bytes(data)`` — serialization
- Static: ``add``, ``sub``, ``hadamard``, ``dot``, ``reduce``


In [ ]:
import mpmt
from array import array

Rv = mpmt.Rvector(ell=4)
v = Rv(size=100)
assert v.size == 100

v[0] = 5
v[1] = 12
assert v[0] == 5 and v[1] == 12

v.fill(val=7)
assert v[0] == 7

indices = array("Q", [0, 1, 2])
v.batch_set(indices=indices, val=3)
assert v[0] == 3

data = v.to_bytes()
v2 = Rv(size=100)
v2.from_bytes(arg=data)
assert v == v2

print("rvector: OK")


**File I/O**: ``v.save(path, auxBuf)`` / ``v.load(path, auxBuf)`` — compact
bit-packed disk persistence (``.mpmtrvp`` format).  Requires a pre-allocated
``RvectorPack`` scratch buffer:

In [ ]:
import tempfile, os
import mpmt

Rv = mpmt.Rvector(ell=4)
v = Rv(size=1_000_000)
v.rand_fill()

# Pre-allocate a pack buffer — must match ell and n
aux = mpmt.RvectorPack(ell=4)(n=1_000_000)

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "demo.mpmtrvp")

    # Save — ELL=4 uses SIMD nibble-packing (2:1 compression)
    v.save(path, aux)

    fsize_mb = os.path.getsize(path) / 1e6
    raw_mb = v.size / 1e6
    print(f"raw={raw_mb:.1f} MB  packed={fsize_mb:.1f} MB  ratio={raw_mb/fsize_mb:.1f}:1")

    # Load into a fresh vector
    v2 = Rv(size=1_000_000)
    v2.load(path, aux)
    assert v == v2
    print("round-trip: OK")

## Arithmetic

All arithmetic is **static** (operates on plain ``Rvector`` values,
no shares).  ``out`` must be pre-allocated and **must not alias**
``a`` or ``b`` — the library does not check for aliasing.

### Vector-Vector

| Method | Signature | Semantics |
|--------|-----------|-----------|
| ``add`` | ``(a, b, out)`` | ``out = a + b  (mod 2^ELL)`` |
| ``sub`` | ``(a, b, out)`` | ``out = a - b  (mod 2^ELL)`` |
| ``hadamard`` | ``(a, b, out)`` | ``out[i] = a[i]·b[i]  (mod 2^ELL)`` |

### Vector-Scalar

| Method | Signature | Semantics |
|--------|-----------|-----------|
| ``add_scalar`` | ``(a, scalar, out)`` | ``out = a + scalar  (mod 2^ELL)`` |
| ``sub_scalar`` | ``(a, scalar, out)`` | ``out = a - scalar  (mod 2^ELL)`` |
| ``mul_scalar`` | ``(a, scalar, out)`` | ``out = a * scalar  (mod 2^ELL)`` |

### Reductions

| Method | Signature | Returns |
|--------|-----------|---------|
| ``dot`` | ``(a, b)`` | ``Σ a[i]·b[i]  mod 2^ELL`` |
| ``reduce`` | ``(a)`` | ``Σ a[i]  mod 2^ELL`` |

In [ ]:
Rv = mpmt.Rvector(ell=4)

# ——— Vector-Vector ———
a = Rv(100); b = Rv(100); out = Rv(100)
a.fill(3); b.fill(2)
Rv.add(a, b, out)              # out[i] = 5
Rv.sub(a, b, out)              # out[i] = 1
Rv.hadamard(a, b, out)         # out[i] = 3*2 = 6

# ——— Vector-Scalar ———
Rv.add_scalar(a, 1, out)       # out[i] = 4
Rv.sub_scalar(a, 1, out)       # out[i] = 2
Rv.mul_scalar(a, 3, out)       # out[i] = 9

# ——— Reductions ———
d = Rv.dot(a, b)               # Σ 3*2 = 600 mod 16 = 8
s = Rv.reduce(a)               # Σ 3 = 300 mod 16 = 12
print(f"dot={d}, reduce={s}")